### C3M10: Data Cleaning Pt. 1

## Data Preparation and Cleaning

General principles for starting work on a data set:

1)  Get the data
2)  Observe a few rows (observations)
3)  Get the dimensions of the dataset (how many observations, how many variables)
4)  Get the know the variables - data types, range of values, missing values
5)  Reduce the dataset to variables and observations of interest
6)  Make plots / tables of variables of interest to look for outliers, problems, etc.
7)  Start analysis . .
...

117)  Clean the data more when you discover problems!
118)  More Analysis

## CASE STUDY for today

We will look at data from [National Water quality Monitoring Council](https://www.waterqualitydata.us/).  
In particular, I downloaded Physical Chemistry measurements from Hartford County, CT.

What follows is also an example of one solution to dealing with large datasets.  The entire dataset is about 200MB which is larger than most file sharing locations allow.  I divided the data into 10 pieces, each about 20MB, read in the data, and then recombined into a single datasetset.


In [ ]:
#This is to set global options so we can actually see all columns of a dataset when we ask for them
options(repr.matrix.max.cols = 150, repr.matrix.max.rows = 200)
library(dplyr)
library(ggplot2)


In [ ]:
#get data files
water0 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_0.csv", header = T)
water1 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_1.csv", header = T)
water2 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_2.csv", header = T)
water3 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_3.csv", header = T)
water4 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_4.csv", header = T)
water5 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_5.csv", header = T)
water6 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_6.csv", header = T)
water7 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_7.csv", header = T)
water8 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_8.csv", header = T)
water9 <- read.csv("https://raw.githubusercontent.com/jreuning/YSE_EDA_data/refs/heads/main/PhysChemHartford_9.csv", header = T)
head(water0)

In [ ]:
#Make the final dataset
water <- rbind(water0, water1, water2, water3, water4, water5, water6, water7, water8, water9)

In [ ]:
#How big is the dataset?
dim(water)

#What are the variable names?
names(water)

#Look at the first few rows
head(water)

There are lots of directions we could go.

Things I noticed about the variables upon first inspection:
 
1)  Information on location, time, method, organization related to the collection
2)  Information on type of collection and conditions of collection
3)  Information on multiple characteristics (one per row - Temperature, Salinity, etc.)

LOTS of data here.  One of the first things to do is focus on data of interest.

I decided to start reducing by focusing on the following:

* Dates between year 2000 and the present
* Focus on a few water quality characteristics

**SO** - first we need to identify and clean dates.  It appears we have a variable called "ActivityStartDate"

In [ ]:
#check out a few values
water$ActivityStartDate[1:100]

#What is the variable type?
typeof(water$ActivityStartDate)

We will need to convert this to type 'Date'.  There are several ways to do this - here is one using `as.Date()`.

In [ ]:
# Make a new column that is in date format using as.Date
water$date <- as.Date(water$ActivityStartDate, format = "%m/%d/%Y")

#Check structure
str(water$date)

# Get the range (min and max dates)
range(water$date)

Range is 1915 to 2024.  We make `water2` which has data from 2000 to the present.

In [ ]:
#Make a new dataset called water2 that is our reduced version.  We choose rows that meet the condition outlined.
water2 <- water[water$date  > "2000-01-01", ]

#get new dimensions
dim(water2)

#confirm we got the correct dates
range(water2$date)


**NEXT** - we examine what water characteristics are available. I am guessing in our list of variables that we want "CharacteristicName".

In [ ]:
#What is the data type of Characteristic Name?
typeof(water2$CharacteristicName)

#What are unique values and how often to they occur?
#table(water2$CharacteristicName)
table(water2$CharacteristicName)[1:10]

This is a great example of discovering a complicated mess and then making updates.

Let's get a list of unique Physical/Chemical characteristics that exist.

In [ ]:
#Get unique values
#sort(unique(water2$CharacteristicName))
sort(unique(water2$CharacteristicName))[1:20]
#How long is this list?
length(unique(water2$CharacteristicName))


This is still a lot to digest.  What are the 20 most commonly measured characteristics?

In [ ]:
sort(table(water2$CharacteristicName), decreasing = T)[1:20]

I was curious how many types of measurements contained "Nitr" or "nitr".  Remember, R is case sensitive.

A cool function to see if a word is contained in a text character is 'grep()'.

In [ ]:
#example of grep
quickvec <- c("Airplane", "airplane", "boat", "air", "Hair", "zebra")
grep("air", quickvec)



In [ ]:
#look for uppercase as well - remember that | means 'or'
grep("air|Air", quickvec)

#This can also be done by making all lowercase using the tolower() function
tolower(quickvec)
grep("air", tolower(quickvec))

Let's look for 'Nitr' now

In [ ]:
#Look for Nitr or nitr
temp <- grep("Nitr|nitr", water2$CharacteristicName)
temp[1:100]
unique(water2$CharacteristicName[temp])

Back to data reduction:   I decide to focus on three characteristics for the moment: 

* Nitrogen, mixed forms (NH3), (NH4), organic, (NO2) and (NO3)
* Temperature, water
* Oxygen

Let's updated our dataset to include only rows that correspond to these measurements:

In [ ]:
#Make a list of retained characteristics
charVec <- c("Nitrogen, mixed forms (NH3), (NH4), organic, (NO2) and (NO3)",
             "Temperature, water",
             "Oxygen")

#update water 2 to contain only rows that match this characteristic
water2 <- water2[water2$CharacteristicName %in% charVec, ]
dim(water2)

We're down to 11k observations from > 400k.

We start by making a dataset that just has the information about Nitrogen.

In [ ]:
#Make a dataset that is just nitrogen values
nitrogen <- water2[water2$CharacteristicName == "Nitrogen, mixed forms (NH3), (NH4), organic, (NO2) and (NO3)", ]
dim(nitrogen)
#Note this is same value we saw in table before

Which column has the actual measured values of nitrogen?

In [ ]:
#names(nitrogen)
head(nitrogen, 4)
names(nitrogen)
nitrogen$ResultMeasureValue[1:100]

OK, so this is a character variable, not numeric.

Also, we better see if the units are all the same.

In [ ]:
table(nitrogen$ResultMeasure.MeasureUnitCode)

This means that for 55 measurements, we don't know the units.  Let's look at the first few rows of the missing values.

In [ ]:
nitrogen[nitrogen$ResultMeasure.MeasureUnitCode == "", ][1:3, ]


It looks like unusual things happened for these data points (see comments and note that all measurement values are missing)

Let's update nitrogen to exclude values with no units

In [ ]:
nitrogen <- nitrogen[nitrogen$ResultMeasure.MeasureUnitCode != "", ]
dim(nitrogen)

Back to nitrogen values which are character

In [ ]:
head(sort(nitrogen$ResultMeasureValue))
tail(sort(nitrogen$ResultMeasureValue))

#Looks like they are all numbers, so we convert to numeric and add a new column with this data
nitrogen$Nitrogen <- as.numeric(nitrogen$ResultMeasureValue)
#Look at first 10 values
nitrogen$Nitrogen[1:10]

Let's get a quick look at nitrogen values

In [ ]:
hist(nitrogen$Nitrogen, col = "Green",
     main = "Nitrogen Values",
     xlab = "mg/ml")

#Just out of curiosity, try log scale.  NOTE  - we didn't have zero values, but it is important to check this

hist(log(nitrogen$Nitrogen), col = "Green",
     main = "log(Nitrogen Values)",
     xlab = "log(mg/ml)")

Let's look at variables again - I was curious about Subdivision Name

In [ ]:

table(nitrogen$ActivityMediaSubdivisionName)


In [ ]:
#Compare Groundwater to Surface Water, raw and log scale
boxplot(Nitrogen ~ ActivityMediaSubdivisionName, data = nitrogen,
        col = c("green", "blue"),
        xlab = "",
        main = "Nitrogen by Measurement Type")

boxplot(log(Nitrogen) ~ ActivityMediaSubdivisionName, data = nitrogen,
        col = c("green", "blue"),
        xlab = "",
        main = "Nitrogen by Measurement Type")

It appears that Groundwater values are generally higher.

I was concerned by two other columns:

* ResultSampleFractionText
* USGSPCode

I started by making a table of the levels and counts for each column

In [ ]:
table(nitrogen$USGSPCode)
table(nitrogen$ResultSampleFractionText)

A quick google search  took me back to the original website where I discovered there is a site with United States Geological Survey Parameter codes (USGSP).  I looked up 'Nutrient Parameter Codes' and discovered on their [website](https://help.waterdata.usgs.gov/parameter_cd?group_cd=NUT) that 


*  600 is Total Nitrogen, used by the EPA

*  602 is Dissolved Nitrogen, used by the EPA

*  62854 is Dissolved but analytically determined, NOT used by the EPA

*  62855 is Total but analytically determined, NOT used by the EPA 

This is concerning in that it looks like I actually may be combining 4 different measurements together and pretending they are a single measurement.

Let's look at our boxplots again (log scale) by USGSPCode.


In [ ]:
boxplot(log(Nitrogen) ~ USGSPCode, data = nitrogen,
        col = c("green", "blue"),
        xlab = "",
        main = "Nitrogen by Measurement Type")

Let's update nitrogen to JUST be total (not dissolved), and we'll remove groundwater

In [ ]:
nitrogen <- nitrogen[nitrogen$ActivityMediaSubdivisionName != "Groundwater" & nitrogen$USGSPCode %in% c(600, 62855), ]
dim(nitrogen)

NOW: we are finally ready to start looking at total nitrogen.
We start by looking at how nitrogen has changed over time.

In [ ]:
plot(log(nitrogen$Nitrogen) ~ nitrogen$date,
     col = "blue",
     pch = 19,
     main = "log(Nitrogen) vs. Time",
     xlab = "Year",
     ylab = "log(mg/ml)")

Seems as if there are two stripes.  This is interesting and maybe concerning (are there still two groups?)
What about looking at levels vs. month of year?  Maybe there is seasonality?

Here is example of how to use AI to get code:

Question: write r code that takes a variable in Date format and creates a new column that is date but in order of months from January through December


In [ ]:
# Create a new column with month names ordered from January to December
nitrogen$month_ordered <- factor(format(nitrogen$date, "%B"),
                           levels = month.name,
                           ordered = TRUE)

#Look at first 50 values
nitrogen$month_ordered[1:50]

#Make new plot
boxplot(log(nitrogen$Nitrogen) ~ nitrogen$month_ordered,
     col = "blue",
     pch = 19,
     main = "log(Nitrogen) vs. Time",
     xlab = "Month",
     ylab = "log(mg/ml)",
     las = 2)

So it appears that the differences we were seeing were not due to month.

I invstigated further to see if this was related to Hydrologic Condition or Hydrologic Event.


In [ ]:
table(nitrogen$HydrologicCondition)
table(nitrogen$HydrologicEvent)

In [ ]:
#Make new plot
boxplot(log(nitrogen$Nitrogen) ~ nitrogen$HydrologicEvent,
     col = "blue",
     pch = 19,
     main = "log(Nitrogen) vs. Time",
     xlab = "Month",
     ylab = "log(mg/ml)",
     las = 2)

It seems like this might be related to the bands that we're seeing in our scatterplot.

We update the scatterplot to use a different color for each Hydrologic Event.

In [ ]:
plot(log(Nitrogen) ~ date,
     data = nitrogen,
     col = as.factor(HydrologicEvent),
     pch = 19,
     main = "log(Nitrogen) vs. Time",
     xlab = "Year",
     ylab = "log(mg/ml)",
     ylim = c(-2.5, 3))
legend("topright", col = 1:5, legend =  levels(as.factor(nitrogen$HydrologicEvent)), pch = 19, cex = .7)

In [ ]:
It looks like there might be some relationship here, but more investigation is required!


## Oxygen vs. Temperature
We now investigate the relationship between oxygen levels and water temperature.  It is well known that oxygen capacity decreases as temperature increases.

We'll need to make a new dataset with oxygen and temperature AND we need to then merge and make sure we're merging correctly.

From last time, I learned to check USGSP Codes!

* 300 is Dissolved oxygen, ml/l
* 301 is Dissolved oxygen, percent of saturation

We will use only dissolved oxygen in terms of ml/l.


In [ ]:
#Make oxygen dataframe
oxygen <- water2[water2$CharacteristicName == "Oxygen" & water2$USGSPCode == 300, ]
dim(oxygen)
head(oxygen)

Now we need to find corresponding water temperature values

In [ ]:
tempC <- water2[water2$CharacteristicName == "Temperature, water", ]
head(tempC)
#confirm all one USGSP code - the code we want is 10
table(tempC$USGSPCode)

dim(tempC)
dim(oxygen)

**NOW** - we want to merge these datasets together.

We will need to merge by data/time to make sure they match.  We'll also want to remove most other variables to keep things simpler.


In [ ]:
head(oxygen)
oxygen <- oxygen[, c("ActivityStartDate", "ActivityStartTime.Time", "MonitoringLocationIdentifier", "ResultMeasureValue", "ActivityMediaSubdivisionName")]
oxygen <- oxygen |> rename(Oxygen = ResultMeasureValue)

tempC <- tempC[, c("ActivityStartDate", "ActivityStartTime.Time", "MonitoringLocationIdentifier", "ResultMeasureValue")]
tempC <- tempC |> rename(TempC = ResultMeasureValue)

dim(oxygen)
dim(tempC)

In [ ]:
#merge - by default, it will delete observations that are not in both datasets
newData <- merge(oxygen, tempC)
dim(newData)

**GREAT**.  Now we can make a scatterplot.

In [ ]:
plot(Oxygen ~ TempC,
     data = newData,
     col = "blue",
     pch = 19,
     main = "Oxygen vs. Temperature",
     xlab = "Temp (C)",
     ylab = "mg/l")

I was curious about Oxygen levels of 5 or less. This made we wonder if some oxygen values were collected under a different condition.

Back to the oxygen data:

In [ ]:
head(newData)
table(newData$ActivityMediaSubdivisionName)

We update our plot.

In [ ]:
plot(Oxygen ~ TempC,
     data = newData,
     col = as.factor(ActivityMediaSubdivisionName),
     pch = 19,
     main = "Oxygen vs. Time",
     xlab = "Temp (C)",
     ylab = "mg/l")
legend("topright", col = 1:3, legend =  levels(as.factor(newData$ActivityMediaSubdivisionName)), pch = 19)

From here: there are so many other possible analyses you could do.  Each variable we consider will require different data cleaning and different analysis.

Welcome to the Data Cleaning Club!
